In [33]:
import os
import json
import pandas as pd
import traceback

In [34]:
from langchain.chat_models import ChatOpenAI

In [35]:
from dotenv import load_dotenv

load_dotenv()  # take environment variables from .env.

True

In [36]:
KEY=os.getenv("OPENAI_API_KEY")

In [37]:
llm=ChatOpenAI(openai_api_key=KEY,model_name="gpt-3.5-turbo", temperature=0.5)

In [38]:
# Create ChatOpenAI instance using KEY
if KEY:
    llm = ChatOpenAI(openai_api_key=KEY, model_name="gpt-3.5-turbo", temperature=0.5)
    print("llm created:", type(llm).__name__)
else:
    print("KEY is not set. Please set OPENAI_API_KEY in your .env or assign KEY variable.")


llm created: ChatOpenAI


In [39]:
print(KEY)

<REDACTED>


In [40]:
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000021F1DD44400>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000021F1DD8D910>, temperature=0.5, openai_api_key='<REDACTED>', openai_proxy='')

In [41]:
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains import SequentialChain
from langchain.callbacks import get_openai_callback
import PyPDF2

In [42]:
# Additional imports requested by user
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, SequentialChain
from langchain.callbacks import get_openai_callback
import PyPDF2
print('import cell added')


import cell added


In [43]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [44]:
TEMPLATE="""
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [45]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
    )

In [46]:
quiz_chain=LLMChain(llm=llm, prompt=quiz_generation_prompt, output_key="quiz", verbose=False)

In [47]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [48]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject", "quiz"], template=TEMPLATE2)

In [49]:
review_chain=LLMChain(llm=llm, prompt=quiz_evaluation_prompt, output_key="review", verbose=False)

In [50]:
generate_evaluate_chain=SequentialChain(
    chains=[quiz_chain, review_chain],
    input_variables=["text", "number", "subject", "tone", "response_json"],
    output_variables=["quiz", "review"],
    verbose=False,
)

In [51]:

file_path=r"C:\Users\user\mcqgen\mcqgenrator.egg-info\data.txt"

In [52]:
file_path

'C:\\Users\\user\\mcqgen\\mcqgenrator.egg-info\\data.txt'

In [53]:
with open(file_path, 'r') as file:
    TEXT = file.read()


In [54]:
print(TEXT)

The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[5][6] The synonym self-teaching computers was also used during this time period.[7][8]

The earliest machine learning program was introduced in the 1950s, when Samuel invented a computer program that calculated the chance of winning in checkers for each side, but the history of machine learning is rooted in decades of efforts to study human cognitive processes.[9] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[10] The Hebbian theory of neuron interaction set the groundwork for how many machine learning algorithms work, with connected artificial neurons changing the strength of their connections based on data.[9] Other researchers who have studied human cognitive systems contributed to the m

In [55]:
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [56]:
NUMBER=5 
SUBJECT="biology"
TONE="simple"

In [64]:
# Full MCQ generation execution with OpenAI callback metrics
from openai import RateLimitError


def fallback_generate_quiz(text, number, subject, tone):
    sentences = [s.strip() for s in text.replace("?", ".").replace("!", ".").split(".") if s.strip()]
    fallback = {str(i + 1): {
        "mcq": f"Based on the passage, what is the main idea of sentence {min(i, len(sentences)-1)+1}?",
        "options": {
            "a": sentences[min(i, len(sentences)-1)] if len(sentences) > i else "A key idea from the passage.",
            "b": sentences[min(i+1, len(sentences)-1)] if len(sentences) > i+1 else "Another possible idea.",
            "c": sentences[min(i+2, len(sentences)-1)] if len(sentences) > i+2 else "A related idea.",
            "d": "None of the above",
        },
        "correct": "a",
    } for i in range(min(number, 5))}
    return {
        "quiz": fallback,
        "review": (
            f"Fallback quiz generated for {subject} in {tone} tone using the provided text. "
            "This output is a local placeholder because the OpenAI quota was unavailable."
        ),
    }

try:
    with get_openai_callback() as cb:
        response = generate_evaluate_chain(
            {
                "text": TEXT,
                "number": NUMBER,
                "subject": SUBJECT,
                "tone": TONE,
                "response_json": json.dumps(RESPONSE_JSON),
            },
            callbacks=[cb],
        )
    print("callback metrics:", cb)
    print("Total Tokens:", cb.total_tokens)
    print("Prompt Tokens:", cb.prompt_tokens)
    print("Completion Tokens:", cb.completion_tokens)
    print("Total Cost:", cb.total_cost)
    print("response type:", type(response))
    print("response:", response)

    if response is None:
        raise RuntimeError("Chain execution returned None. Check the OpenAI call and retry.")
    if not isinstance(response, dict):
        raise RuntimeError(f"Expected response dict, got {type(response).__name__}: {response!r}")

except RateLimitError as e:
    print("OpenAI request failed due to quota or rate limiting:", e)
    print("Using local fallback output instead of OpenAI.")
    response = fallback_generate_quiz(TEXT, NUMBER, SUBJECT, TONE)

quiz = response.get("quiz")
review = response.get("review")
print("quiz output:", quiz)
print("review output:", review)

OpenAI request failed due to quota or rate limiting: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Using local fallback output instead of OpenAI.
quiz output: {'1': {'mcq': 'Based on the passage, what is the main idea of sentence 1?', 'options': {'a': 'The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence', 'b': '[5][6] The synonym self-teaching computers was also used during this time period', 'c': '[7][8]\n\nThe earliest machine learning program was introduced in the 1950s, when Samuel invented a computer program that calculated the chance of winning in checkers for each side, but the history of machine learning is rooted in dec